In [ ]:
import pandas as pd
from typing import List
from pydantic import BaseModel, Field
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

OPEN_AI_KEY = "chave_aqui"

In [31]:
df = pd.read_csv("data/processed/dataset.csv")
df = df[['texto_limpo', 'classe_macro', 'classe_detalhada']]
df

,texto_limpo,classe_macro,classe_detalhada
0,erro recorrente sistema tentar acessar determi...,Problemas Técnicos,Erro de Sistema / Aplicação
1,pedido integracao sincronizacao dados,Solicitações Operacionais,Integração com Sistemas Externos
2,aguardando posicionamento,Outros,Mensagem Genérica
3,sistema apresentou erro inesperado executar fu...,Problemas Técnicos,Erro de Sistema / Aplicação
4,registro automatico sistema,Outros,Registro Automático
...,...,...,...
73,solicito treinamento equipe,Solicitações Operacionais,Solicitação de Treinamento
74,erro gerar relatorio financeiro pdf csv,Problemas Técnicos,Erro de Exportação / Relatórios
75,necessario capacitacao usuarios sistema,Solicitações Operacionais,Solicitação de Treinamento
76,valor fatura parece incorreto,Financeiro e Cobrança,Dúvidas sobre Fatura / Valores


In [37]:
X = df["texto_limpo"]
y_macro = df["classe_macro"]
y_detail = df["classe_detalhada"]

X_train, X_test, y_macro_train, y_macro_test, y_detail_train, y_detail_test = train_test_split(
    X, y_macro, y_detail, test_size=0.2, random_state=42, stratify=y_macro
)

In [32]:
macro_classes = sorted(df["classe_macro"].unique())

macro_to_details = {
    macro: sorted(df[df["classe_macro"] == macro]["classe_detalhada"].unique())
    for macro in macro_classes
}

In [40]:
class ClassificationOutput(BaseModel):
    macro_primary: str = Field(description="Classe macro principal, obrigatoriamente uma das macros fornecidas")
    macro_secondary: List[str] = Field(default_factory=list, description="Lista de macros secundárias possíveis")
    detail_primary: str = Field(description="Classe detalhada principal, obrigatoriamente pertencente à macro principal")
    detail_secondary: List[str] = Field(default_factory=list, description="Lista de classes detalhadas secundárias possíveis")

In [41]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPEN_AI_KEY)
structured_llm = llm.with_structured_output(ClassificationOutput, method="json_schema")

In [42]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Você é um classificador hierárquico de textos. "
        "Classifique somente com classes válidas fornecidas. "
        "A classe detalhada principal deve pertencer à macro principal. "
        "Se houver ambiguidade, mantenha uma classe principal e registre alternativas nas listas secundárias. "
        "Nunca invente classes."
    ),
    (
        "human",
        """Classes macro válidas:
{macro_classes}

Hierarquia macro -> detalhadas:
{hierarchy}

Texto:
{texto}"""
    )
])

In [43]:
chain = prompt | structured_llm

In [44]:
results = []

for text in tqdm(X_test.tolist()):
    try:
        res = chain.invoke({
            "macro_classes": macro_classes,
            "hierarchy": macro_to_details,
            "texto": text
        })
        results.append({
            "macro_primary": res.macro_primary,
            "macro_secondary": res.macro_secondary,
            "detail_primary": res.detail_primary,
            "detail_secondary": res.detail_secondary
        })
    except:
        results.append({
            "macro_primary": None,
            "macro_secondary": [],
            "detail_primary": None,
            "detail_secondary": []
        })

100%|██████████| 16/16 [00:21<00:00,  1.32s/it]


In [45]:
pred_df = pd.DataFrame(results)

pred_df["macro_true"] = y_macro_test.values
pred_df["detail_true"] = y_detail_test.values

In [48]:
valid_macro = pred_df.dropna(subset=["macro_primary"]).copy()
print(classification_report(valid_macro["macro_true"], valid_macro["macro_primary"], zero_division=0))

                           precision    recall  f1-score   support

   Feedback e Experiência       0.50      0.50      0.50         2
    Financeiro e Cobrança       1.00      1.00      1.00         4
                   Outros       1.00      0.50      0.67         2
       Problemas Técnicos       0.80      1.00      0.89         4
Solicitações Operacionais       1.00      1.00      1.00         4

                 accuracy                           0.88        16
                macro avg       0.86      0.80      0.81        16
             weighted avg       0.89      0.88      0.87        16



In [47]:
valid_detail = pred_df.dropna(subset=["detail_primary"]).copy()
print(classification_report(valid_detail["detail_true"], valid_detail["detail_primary"], zero_division=0))

                                        precision    recall  f1-score   support

                 Agradecimento Simples       1.00      1.00      1.00         1
                 Atualização Cadastral       1.00      1.00      1.00         1
          Criação / Gestão de Usuários       0.50      1.00      0.67         1
            Customização de Relatórios       0.00      0.00      0.00         1
        Dúvidas sobre Fatura / Valores       1.00      1.00      1.00         1
           Erro de Sistema / Aplicação       1.00      1.00      1.00         2
               Falha de Banco de Dados       0.00      0.00      0.00         1
  Indisponibilidade / Queda do Serviço       0.00      0.00      0.00         0
Pagamento Realizado mas Não Compensado       1.00      1.00      1.00         1
   Problemas de Performance / Lentidão       0.50      1.00      0.67         1
                   Reembolso / Estorno       1.00      1.00      1.00         1
                   Registro Automático 

In [49]:
pred_df.head()

,macro_primary,macro_secondary,detail_primary,detail_secondary,macro_true,detail_true
0,Solicitações Operacionais,[],Atualização Cadastral,[],Solicitações Operacionais,Atualização Cadastral
1,Solicitações Operacionais,[],Reprocessamento / Correção de Dados,[],Solicitações Operacionais,Reprocessamento / Correção de Dados
2,Solicitações Operacionais,[],Criação / Gestão de Usuários,"[Alteração de Plano / Contrato, Atualização Ca...",Solicitações Operacionais,Customização de Relatórios
3,Financeiro e Cobrança,[],Pagamento Realizado mas Não Compensado,[],Financeiro e Cobrança,Pagamento Realizado mas Não Compensado
4,Financeiro e Cobrança,[],Dúvidas sobre Fatura / Valores,[],Financeiro e Cobrança,Dúvidas sobre Fatura / Valores
